# Session 4: Similarity & Clustering — Finding Hidden Groups

## The Story

Your marketing team has been using the same 3 customer segments for 10 years: "Young Drivers," "Families," and "Seniors."

Let's see what happens when we let the DATA define the segments instead.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

In [ ]:
# Create a realistic customer portfolio with natural clusters
n = 3000

# Cluster 1: Urban professionals (young, high premium, moderate km)
n1 = 700
c1 = pd.DataFrame({
    'age': np.random.normal(32, 5, n1).clip(22, 45).astype(int),
    'annual_km': np.random.normal(12000, 3000, n1).clip(3000, 25000).astype(int),
    'vehicle_value': np.random.normal(35000, 10000, n1).clip(15000, 65000).astype(int),
    'annual_premium': np.random.normal(900, 200, n1).clip(400, 1600).astype(int),
    'tenure_years': np.random.exponential(2, n1).clip(0, 10).astype(int),
    'claims_3y': np.random.poisson(0.7, n1),
    'n_policies': np.random.choice([1, 2], n1, p=[0.7, 0.3]),
})

# Cluster 2: Suburban families (mid-age, moderate everything, multi-policy)
n2 = 1000
c2 = pd.DataFrame({
    'age': np.random.normal(42, 6, n2).clip(30, 55).astype(int),
    'annual_km': np.random.normal(18000, 4000, n2).clip(8000, 35000).astype(int),
    'vehicle_value': np.random.normal(25000, 7000, n2).clip(12000, 45000).astype(int),
    'annual_premium': np.random.normal(700, 150, n2).clip(350, 1200).astype(int),
    'tenure_years': np.random.exponential(5, n2).clip(1, 20).astype(int),
    'claims_3y': np.random.poisson(0.4, n2),
    'n_policies': np.random.choice([1, 2, 3], n2, p=[0.2, 0.5, 0.3]),
})

# Cluster 3: Rural retirees (older, low km, cheap vehicles)
n3 = 600
c3 = pd.DataFrame({
    'age': np.random.normal(65, 7, n3).clip(55, 80).astype(int),
    'annual_km': np.random.normal(6000, 2000, n3).clip(1000, 12000).astype(int),
    'vehicle_value': np.random.normal(15000, 5000, n3).clip(5000, 30000).astype(int),
    'annual_premium': np.random.normal(500, 100, n3).clip(250, 800).astype(int),
    'tenure_years': np.random.exponential(8, n3).clip(3, 25).astype(int),
    'claims_3y': np.random.poisson(0.3, n3),
    'n_policies': np.random.choice([1, 2], n3, p=[0.6, 0.4]),
})

# Cluster 4: Road warriors (mid-age, very high km, commercial-adjacent)
n4 = 700
c4 = pd.DataFrame({
    'age': np.random.normal(38, 8, n4).clip(25, 55).astype(int),
    'annual_km': np.random.normal(35000, 8000, n4).clip(20000, 60000).astype(int),
    'vehicle_value': np.random.normal(20000, 6000, n4).clip(8000, 40000).astype(int),
    'annual_premium': np.random.normal(1100, 250, n4).clip(500, 2000).astype(int),
    'tenure_years': np.random.exponential(3, n4).clip(0, 15).astype(int),
    'claims_3y': np.random.poisson(0.8, n4),
    'n_policies': np.random.choice([1, 2], n4, p=[0.8, 0.2]),
})

data = pd.concat([c1, c2, c3, c4], ignore_index=True)

print("📋 CUSTOMER PORTFOLIO")
print("=" * 50)
print(f"  Total customers: {len(data):,}")
print(f"\n  Feature summary:")
print(data.describe().round(0).to_string())

## Part 1: Finding Nearest Neighbors

Before clustering, let's understand **similarity.** Given a specific customer, who are the most similar ones?

In [ ]:
# Normalize features (so age and km are on comparable scales)
features = ['age', 'annual_km', 'vehicle_value', 'annual_premium', 'tenure_years', 'claims_3y', 'n_policies']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(data[features])

# Find nearest neighbors for a specific customer
nn = NearestNeighbors(n_neighbors=6)  # 5 neighbors + the customer itself
nn.fit(X_scaled)

# Pick a customer and find their nearest neighbors
customer_idx = 42
distances, indices = nn.kneighbors(X_scaled[customer_idx:customer_idx+1])

print("📊 NEAREST NEIGHBOR SEARCH")
print("=" * 70)
print(f"\n  TARGET CUSTOMER (#{customer_idx}):")
print(f"  {data.iloc[customer_idx][features].to_dict()}")
print(f"\n  5 MOST SIMILAR CUSTOMERS:")
print(f"  {'Rank':<6} {'Distance':<10} {'Age':<6} {'km/yr':<8} {'Value':<10} {'Premium':<10} {'Tenure':<8} {'Claims':<8} {'Policies'}")
print("  " + "-" * 75)
for rank, (idx, dist) in enumerate(zip(indices[0][1:], distances[0][1:])):
    row = data.iloc[idx]
    print(f"  {rank+1:<6} {dist:<10.2f} {row['age']:<6} {row['annual_km']:<8} €{row['vehicle_value']:<9,} €{row['annual_premium']:<9,} {row['tenure_years']:<8} {row['claims_3y']:<8} {row['n_policies']}")

print(f"\n  💡 These are the customers most 'like' customer #{customer_idx}.")
print(f"     If 4 out of 5 lapsed last year, this customer is probably at risk too.")

---

## Part 2: Clustering — Let the Data Define Segments

Instead of pre-defining "Young/Family/Senior," let's ask: **what groups does the data naturally contain?**

In [ ]:
# How many clusters? Try several values of k
inertias = []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(K_range, inertias, 'o-', color='#00008f', lw=2.5, markersize=8)
ax.set_xlabel('Number of Clusters (k)', fontsize=12)
ax.set_ylabel('Within-Cluster Variance (lower = tighter groups)', fontsize=11)
ax.set_title('The Elbow Method: How Many Segments?', fontsize=13, fontweight='bold')
ax.axvline(x=4, color='#FF1721', linestyle='--', alpha=0.7, label='Elbow at k=4')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n→ The 'elbow' suggests 4 natural clusters.")
print("  After 4, adding more clusters gives diminishing returns.")
print("  (This is a guideline, not a hard rule — business sense matters too.)")

In [ ]:
# Build the 4-cluster model
km = KMeans(n_clusters=4, random_state=42, n_init=10)
data['cluster'] = km.fit_predict(X_scaled)

# Profile each cluster
print("📊 DATA-DRIVEN CUSTOMER SEGMENTS")
print("=" * 75)

profiles = data.groupby('cluster')[features].mean().round(0)
profiles['size'] = data.groupby('cluster').size()
profiles['pct'] = (profiles['size'] / len(data) * 100).round(1)

print(profiles.to_string())

print(f"\n💡 The algorithm found 4 groups — let's name them based on their profiles.")

In [ ]:
# Assign business-meaningful names
# Sort clusters by a key characteristic to assign names
cluster_profiles = data.groupby('cluster')[features].mean()

# Determine names based on key characteristics
cluster_names = {}
for c in range(4):
    p = cluster_profiles.loc[c]
    if p['annual_km'] > 25000:
        cluster_names[c] = '🚛 Road Warriors'
    elif p['age'] > 55:
        cluster_names[c] = '🏡 Rural Retirees'
    elif p['age'] < 38 and p['vehicle_value'] > 28000:
        cluster_names[c] = '🏙️ Urban Professionals'
    else:
        cluster_names[c] = '👨‍👩‍👧 Suburban Families'

data['segment_name'] = data['cluster'].map(cluster_names)

print("📊 DISCOVERED SEGMENTS (named by business logic)")
print("=" * 80)
for c in range(4):
    name = cluster_names[c]
    seg = data[data['cluster'] == c]
    print(f"\n  {name}")
    print(f"    Size: {len(seg):,} ({len(seg)/len(data):.0%})")
    print(f"    Avg age: {seg['age'].mean():.0f} | Avg km: {seg['annual_km'].mean():,.0f} | Avg premium: €{seg['annual_premium'].mean():,.0f}")
    print(f"    Avg tenure: {seg['tenure_years'].mean():.1f}y | Avg claims: {seg['claims_3y'].mean():.1f} | Avg policies: {seg['n_policies'].mean():.1f}")

In [ ]:
# Visual: scatter plots showing clusters
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

colors = ['#00008f', '#FF1721', '#27ae60', '#e67e22']
for c in range(4):
    seg = data[data['cluster'] == c]
    axes[0].scatter(seg['age'], seg['annual_km'], c=colors[c], s=15, alpha=0.5, label=cluster_names[c])
    axes[1].scatter(seg['vehicle_value'], seg['annual_premium'], c=colors[c], s=15, alpha=0.5, label=cluster_names[c])

axes[0].set_xlabel('Customer Age', fontsize=11)
axes[0].set_ylabel('Annual km', fontsize=11)
axes[0].set_title('Segments: Age vs. Driving Distance', fontweight='bold')
axes[0].legend(fontsize=9, markerscale=3)

axes[1].set_xlabel('Vehicle Value (€)', fontsize=11)
axes[1].set_ylabel('Annual Premium (€)', fontsize=11)
axes[1].set_title('Segments: Vehicle Value vs. Premium', fontweight='bold')
axes[1].legend(fontsize=9, markerscale=3)

plt.suptitle('Data-Driven Segments vs. Traditional Age-Based Buckets', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n→ Notice the 'Road Warriors' cluster — this group is invisible in a simple age segmentation.")
print("  They're mid-age (like 'Families') but have completely different behavior (35K+ km/year).")
print("  They need a different product, not the same family offer.")

### 🤔 Discussion

1. **Compare to your current segmentation** — does this reveal a group you're not targeting separately?
2. **Would you act differently** for each segment? If not, merge them.
3. **The "Road Warriors"** — what product or service would you offer them that you wouldn't offer families?

---

## Part 3: Anomaly Detection — The Odd Ones Out

Customers who are far from ALL clusters might be worth investigating:

In [ ]:
# Calculate distance from each point to its nearest cluster center
distances_to_center = km.transform(X_scaled).min(axis=1)
data['anomaly_score'] = distances_to_center

# Show the most anomalous customers
top_anomalies = data.nlargest(8, 'anomaly_score')[features + ['segment_name', 'anomaly_score']]

print("📊 MOST UNUSUAL CUSTOMERS (don't fit any segment well)")
print("=" * 80)
print(top_anomalies.to_string(index=False))

print(f"\n  💡 These customers are outliers — far from any cluster center.")
print(f"     Possible reasons:")
print(f"     • Data entry errors (sanity check the values)")
print(f"     • Genuinely unusual profiles (may need bespoke handling)")
print(f"     • Fraud indicators (legitimate customers don't usually look this unusual)")
print(f"\n     Worth investigating — but not every anomaly is a problem.")

---

## Key Takeaways

1. **Nearest neighbors** find similar historical cases — powerful for case-based reasoning
2. **Clustering reveals natural groups** — often different from traditional segments
3. **Feature normalization is essential** — otherwise large-range features dominate
4. **The number of clusters is a business choice** — guided by data but decided by actionability
5. **Anomalies deserve investigation** — outliers may be errors, unusual cases, or fraud

> The algorithm discovers the groups. **You** name them and decide what to do.